In [35]:
import pandas as pd

# Load the data directly from the correct CSV path
df = pd.read_csv("/workspaces/dsa-lab3/lab3/data/book400k-500k.csv")

# Create your search_list by sampling 50,000 IDs
# We skip saving to a file to avoid that FileNotFoundError
search_list = sorted(df['Id'].sample(n=50000, random_state=42).tolist())

print(f"Success! search_list created with {len(search_list)} records.")
print("You can now run your search benchmark cells below.")

Success! search_list created with 50000 records.
You can now run your search benchmark cells below.


In [36]:
print(df.columns)

Index(['PublishYear', 'Rating', 'RatingDistTotal', 'ISBN', 'RatingDist1',
       'Publisher', 'PublishMonth', 'Id', 'Name', 'Authors', 'RatingDist5',
       'RatingDist4', 'PublishDay', 'RatingDist2', 'pagesNumber',
       'RatingDist3', 'CountsOfReview', 'Language'],
      dtype='str')


In [37]:
df_sorted = df.sort_values("Id").reset_index(drop=True)
search_list = df_sorted["Id"].tolist()

print(f"Data sorted, ready to search through {len(search_list)} records.")

Data sorted, ready to search through 55155 records.


# Task 1

In [38]:
def linear_search(data, target):
    for i in range(len(data)):
        if data[i] == target:
            return i  # Found
    return -1  # Not found

In [39]:
#Testing

# Pick a random ID from your list to search for
target_id = search_list[45000] 

# Run your search
result = linear_search(search_list, target_id)

print(f"Found the book at index: {result}")

Found the book at index: 45000


In [40]:
#Linear search
def linear_search(data, target):
    for i in range(len(data)):
        if data[i] == target:
            return i
    return -1

In [41]:
#Binary search
def binary_search(data, target):
    low, high = 0, len(data) - 1
    while low <= high:
        mid = (low + high) // 2
        if data[mid] == target: return mid
        elif data[mid] < target: low = mid + 1
        else: high = mid - 1
    return -1

In [42]:
#Jump search
import math
def jump_search(data, target):
    n = len(data)
    step = int(math.sqrt(n))
    prev = 0
    while data[min(step, n)-1] < target:
        prev = step
        step += int(math.sqrt(n))
        if prev >= n: return -1
    for i in range(prev, min(step, n)):
        if data[i] == target: return i
    return -1

In [43]:
#Interpolation search
def interpolation_search(data, target):
    low, high = 0, len(data) - 1
    while low <= high and target >= data[low] and target <= data[high]:
        if low == high:
            if data[low] == target: return low
            return -1
        pos = low + int(((float(high - low) / (data[high] - data[low])) * (target - data[low])))
        if data[pos] == target: return pos
        if data[pos] < target: low = pos + 1
        else: high = pos - 1
    return -1

In [44]:
import time

def benchmark_search(search_func, data, target):
    times = []
    for _ in range(3):
        start = time.perf_counter()
        search_func(data, target)
        end = time.perf_counter()
        times.append((end - start) * 1000) # Convert to ms
    return sum(times) / len(times)

# Test sizes required
sizes = [1000, 10000, 50000]
algorithms = [linear_search, binary_search, jump_search, interpolation_search]

for n in sizes:
    subset = search_list[:n]
    target = subset[-1] # Target the last item for worst-case timing
    print(f"\nResults for n={n}:")
    for alg in algorithms:
        avg_time = benchmark_search(alg, subset, target)
        print(f"{alg.__name__}: {avg_time:.6f} ms")


Results for n=1000:
linear_search: 0.052302 ms
binary_search: 0.003545 ms
jump_search: 0.022463 ms
interpolation_search: 0.002446 ms

Results for n=10000:
linear_search: 0.956432 ms
binary_search: 0.003388 ms
jump_search: 0.063158 ms
interpolation_search: 0.001883 ms

Results for n=50000:
linear_search: 3.596749 ms
binary_search: 0.004200 ms
jump_search: 0.133677 ms
interpolation_search: 0.001765 ms


**Specific Input:** A dataset with a massive "outlier" (e.g., [1, 2, 3, 4, 1000000]).

**Explanation:** Interpolation search performs poorly here because the large gap in values makes its position "guess" highly inaccurate, forcing it to behave more like a slow linear search.




**Performance Comparison**

n = 1,000: Interpolation Search wins

n = 10,000: Interpolation Search wins

n = 50,000: Interpolation Search wins

In [45]:
#Bad data set, that performs worse
bad_data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 1000000]
target = 9

binary_time = benchmark_search(binary_search, bad_data, target)
interp_time = benchmark_search(interpolation_search, bad_data, target)

print(f"Binary Search: {binary_time:.6f} ms")
print(f"Interpolation Search: {interp_time:.6f} ms")

Binary Search: 0.001704 ms
Interpolation Search: 0.006726 ms


# Task 2

In [55]:

# BST Implementation
class Node:
    def __init__(self, key, value):
        self.key = key
        self.value = value
        self.left = None
        self.right = None

class BinarySearchTree:
    def __init__(self):
        self.root = None

    def insert(self, key, value):
        if self.root is None:
            self.root = Node(key, value)
        else:
            self._insert_recursive(self.root, key, value)

    def _insert_recursive(self, current, key, value):
        if key < current.key:
            if current.left is None:
                current.left = Node(key, value)
            else:
                self._insert_recursive(current.left, key, value)
        else:
            if current.right is None:
                current.right = Node(key, value)
            else:
                self._insert_recursive(current.right, key, value)

    def search(self, key):
        # Iterative search is usually easier to read for simple lookups
        current = self.root
        while current:
            if key == current.key:
                return current.value
            elif key < current.key:
                current = current.left
            else:
                current = current.right
        return None

    # Recursive Traversals
    def inorder(self):
        result = []
        def walk(node):
            if node:
                walk(node.left)
                result.append(node.key)
                walk(node.right)
        walk(self.root)
        return result

    # BFS using a simple list as a queue
    def bfs(self):
        if not self.root:
            return []
        result = []
        queue = [self.root]
        while queue:
            node = queue.pop(0)
            result.append(node.key)
            if node.left: queue.append(node.left)
            if node.right: queue.append(node.right)
        return result

    # Range Query
    def range_query(self, low, high):
        found_values = []
        def _search_range(node):
            if not node:
                return
            # If current key is greater than low, there might be more in the left branch
            if node.key > low:
                _search_range(node.left)
            # Check if current node is in range
            if low <= node.key <= high:
                found_values.append(node.value)
            # If current key is less than high, there might be more in the right branch
            if node.key < high:
                _search_range(node.right)
        _search_range(self.root)
        return found_values

# Part 2: Loading Data & Building the Tree

import sys

# 1. Increase the recursion limit so Python can handle the deeper branches
sys.setrecursionlimit(10000)

# 2. Load the data
df = pd.read_csv("/workspaces/dsa-lab3/lab3/book_sample.csv")
my_data = df.head(50000).copy()

# 3. Double-Shuffle: Shuffle once, then sort by ID and shuffle again 
# to ensure duplicate 'PublishYear' values aren't inserted in a row.
shuffled_data = my_data.sample(frac=1, random_state=42).reset_index(drop=True)

book_tree = BinarySearchTree()

print("Building the tree (this might take a few seconds)...")
for index, row in shuffled_data.iterrows():
    # Per Appendix A: Key is PublishYear
    book_tree.insert(row['PublishYear'], row.to_dict())

print("Done! The tree is built.")

#Part 3: Benchmarking

# Per Appendix A: Range 1990 to 2000
low_year, high_year = 1990, 2000

# 1. BST Range Query Timing
start_time = time.time()
bst_results = book_tree.range_query(low_year, high_year)
bst_duration = time.time() - start_time

# 2. Lazy Baseline (Linear Search) Timing
start_time = time.time()
lazy_results = [row for _, row in my_data.iterrows() if low_year <= row['PublishYear'] <= high_year]
lazy_duration = time.time() - start_time

print("--- Task 2 Results ---")
print(f"BST Query Time: {bst_duration:.6f} seconds")
print(f"Lazy Baseline Time: {lazy_duration:.6f} seconds")
print(f"Total Books Found: {len(bst_results)}")

Building the tree (this might take a few seconds)...
Done! The tree is built.
--- Task 2 Results ---
BST Query Time: 0.009202 seconds
Lazy Baseline Time: 1.949831 seconds
Total Books Found: 15190


In [ ]:
#Test to see if the tree is truly working
print("Sorted years in the tree (first 20):")
print(book_tree.inorder()[:20])

Sorted years in the tree (first 20):
[1851, 1866, 1874, 1899, 1899, 1900, 1900, 1900, 1900, 1900, 1900, 1905, 1910, 1914, 1914, 1920, 1920, 1920, 1922, 1922]
